In [ ]:
import numpy as np
import pandas as pd

### Load datasets for modeling

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

premodel_core = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Core_Final.csv")
premodel_gust = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Gust_Final.csv")

Mounted at /content/drive


/tmp/ipython-input-405/274579847.py:5: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  premodel_gust = pd.read_csv("/content/drive/MyDrive/MDP_Capstone_Project/Data/Model_Gust_Final.csv")


### Quick dataset checks

In [ ]:
print("Core Shape:", premodel_core.shape)
print("Gust Shape:", premodel_gust.shape)
print("Core Class Balance Check:", premodel_core["High_Intensity"].value_counts(normalize=True))
print("Gust Class Balance Check:", premodel_gust["High_Intensity"].value_counts(normalize=True))

Core Shape: (1563080, 21)
Gust Shape: (1011445, 25)
Core Class Balance Check: High_Intensity
0    0.900896
1    0.099104
Name: proportion, dtype: float64
Gust Class Balance Check: High_Intensity
0    0.886772
1    0.113228
Name: proportion, dtype: float64


In [ ]:
premodel_core["LC_Type1"] = premodel_core["LC_Type1"].astype("category")
premodel_gust["LC_Type1"] = premodel_gust["LC_Type1"].astype("category")

print(premodel_core.dtypes)
print(premodel_gust.dtypes)


High_Intensity                     int64
Mean_Temp_C                      float64
Total_Precip_mm                  float64
LC_Type1                        category
Month                              int64
Day_of_Year                        int64
Latitude_Fire                    float64
Longitude_Fire                   float64
Elevation_m                      float64
FRP_max                          float64
Fire_ID                            int64
Climate_ID                        object
LC_Label                          object
Nearest_Core_Station_Dist_km     float64
Core_Dist_75km                      bool
Core_Dist_100km                     bool
Confidence_Ordered                 int64
DayNight                          object
Hour                               int64
Year                               int64
Province                          object
dtype: object
High_Intensity                     int64
Mean_Temp_C                      float64
Total_Precip_mm                  float64
Ma

In [ ]:
# Check how many rows kept if filtering by 100 km
print("Total core rows:", len(premodel_core))
print("Within 100 km:", premodel_core["Core_Dist_100km"].sum())
print("Percent kept:", premodel_core["Core_Dist_100km"].mean())

# Create filtered dataframe
core_100 = premodel_core[premodel_core["Core_Dist_100km"]].copy()
print("Filtered rows:", len(core_100))

# Re-check class balance
print(core_100["High_Intensity"].value_counts(normalize=True))

Total core rows: 1563080
Within 100 km: 1070906
Percent kept: 0.6851255214064539
Filtered rows: 1070906
High_Intensity
0    0.901839
1    0.098161
Name: proportion, dtype: float64


# **Distance-Thresholding Analysis (CORE Dataset)**

## **(1) Define Features and Response**

In [ ]:
# Core feature set for baseline model
core_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year",
    "LC_Type1"
]

X = core_100[core_features]
y = core_100["High_Intensity"]

### **(2) Define Train/Test Split**

In [ ]:
from sklearn.model_selection import train_test_split

# Startified split to ensure rare high-intensity class representation
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y
)

### **(3) Feature Adjustment Preprocessing**

In [ ]:
# Define feature types (numerical and categorical)
numeric_features = [
    "Mean_Temp_C",
    "Total_Precip_mm",
    "Latitude_Fire",
    "Longitude_Fire",
    "Elevation_m",
    "Month",
    "Day_of_Year"
]

categorical_features = ["LC_Type1"]

### Construct preprocessing and model pipeline

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=123
)

pipeline = Pipeline(
    steps=[("preprocess", preprocess), ("model", log_reg)]
)

### **(4) Fit Model & Evaluate Performance**

In [ ]:
# Fit the model
pipeline.fit(X_train, y_train)

# Evaluate Performance
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC–AUC:", roc_auc_score(y_test, y_prob))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.49      0.64    193158
           1       0.12      0.62      0.20     21024

    accuracy                           0.51    214182
   macro avg       0.52      0.56      0.42    214182
weighted avg       0.84      0.51      0.60    214182

ROC–AUC: 0.5826625990330894
Confusion Matrix:
 [[95143 98015]
 [ 7982 13042]]


### **(5) Check Model Coefficients**

In [ ]:
# Extract feature names after preprocessing
feature_names = (
    pipeline.named_steps["preprocess"]
    .get_feature_names_out()
)

coefficients = pipeline.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values("Coefficient", ascending=False)

coef_df.head(10), coef_df.tail(10)

(             Feature  Coefficient
 15   cat__LC_Type1_9     1.038506
 14   cat__LC_Type1_8     0.969990
 16  cat__LC_Type1_10     0.842979
 7    cat__LC_Type1_1     0.692976
 11   cat__LC_Type1_5     0.554557
 10   cat__LC_Type1_4     0.492006
 9    cat__LC_Type1_3     0.435684
 5         num__Month     0.368107
 13   cat__LC_Type1_7     0.262044
 17  cat__LC_Type1_11     0.067495,
                  Feature  Coefficient
 3    num__Longitude_Fire     0.016012
 12       cat__LC_Type1_6    -0.002265
 21      cat__LC_Type1_15    -0.031011
 18      cat__LC_Type1_12    -0.065823
 8        cat__LC_Type1_2    -0.097559
 1   num__Total_Precip_mm    -0.123818
 6       num__Day_of_Year    -0.488054
 20      cat__LC_Type1_14    -0.814282
 22      cat__LC_Type1_16    -2.303123
 19      cat__LC_Type1_13    -2.917455)